# Tutorial 5: PyMC Surrogate Learning (`pymc_gp`)

Estimated time: 30-50 minutes

## Prerequisites
`pymc` and `arviz` in the kernel running this notebook. Two envs work:

- `conda env create -f environment.yml` -> `py314_bayesmm` — **has PyMC *and* SBI**, so T5, T6 and T6's cross-backend comparison all run in one kernel. Recommended for working through the series.
- `conda env create -f environment-pymc.yml` -> `py312_bayesmm_pymc` — PyMC only; enough for T5 alone.

## Learning aims
- Primary package aim: fit/evaluate surrogate models through the CLI and inspect artifacts
- Secondary scientific aim: know what a predictive width is *made of*, and what it does and does not tell you

## Success criteria
- you can train a surrogate, evaluate new inputs, decompose the predictive width into parameter uncertainty + fitted residual scale, and say when the width is a warning and when it is silent

## Why this tutorial matters

A **surrogate** is a fast probabilistic model fit to a sweep's outputs, so you can replace the simulator at inference time — predict at any input without re-running the model. The `pymc_gp` backend returns a *posterior predictive*: every prediction carries a **width** (a posterior standard deviation), not just a point estimate. Learning to read that width is the difference between "the surrogate said 0.42" and "the surrogate said 0.42 with a width of 0.67, which is its way of telling me it cannot represent this function."

**Callback to T1:** at the bottom of T1 you were asked to hold a question — if you fitted a probabilistic surrogate on that 3x3 grid and asked it for **`product` = a*b** at `a = 0.5, b = 0.5`, would you trust the prediction? Steps 1-5 build the machinery on the easy channel (`sum`); **Step 6 asks T1's actual question** and prints the answer. It is not the answer most people expect.

## First, a naming trap: `pymc_gp` is not a Gaussian process

The backend id says `gp`. The implementation is **Bayesian linear regression**
(`src/bayesian_metamodeling/surrogates/backends.py::_fit_pymc_bayesian_linear`):

```
beta      ~ Normal(0, 2)        # one slope per input: beta_a, beta_b
intercept ~ Normal(0, 2)
sigma     ~ HalfNormal(1)
y         ~ Normal(intercept + a*beta_a + b*beta_b, sigma)
```

Four parameters. No kernel, no length-scale, no covariance function — `grep -rn "pm.gp"` over `src/` returns nothing but a comment saying so, and `surrogate_config.py` accepts exactly five keys for this backend (`draws`, `tune`, `chains`, `target_accept`, `output_correlation`), none of them a kernel hyperparameter.

This matters because a GP and a linear model behave in **opposite** ways off the training data:

| | far from the training data |
|---|---|
| Gaussian process | mean falls back toward the prior mean, band opens up — "I don't know" |
| this backend | mean continues the plane forever, band barely moves — "I'm sure" |

So the widely-repeated advice that "a probabilistic surrogate widens away from data and warns you" is **false of the model you are about to fit**. Step 5 measures it instead of asserting it. The general habit is the lesson: check what a backend id actually compiles to before you reason from its name.

## Step 1: Ensure training dataset exists


In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)


# Preflight: detect PyMC. This notebook invokes `bayesmm surrogate fit/eval`
# with the pymc_gp backend, which requires `pymc` in THIS kernel's env. The
# main dev env (py314_bayesmm) deliberately ships without it — the backends
# live in their own envs — so detecting its absence here lets us skip the
# PyMC-specific steps with an actionable banner instead of a RuntimeError
# halfway down the notebook. Mirrors the SBI preflight in Tutorial 6.
import importlib.util as _ilu
import os as _os

PYMC_AVAILABLE = _ilu.find_spec("pymc") is not None

if not PYMC_AVAILABLE:
    _conda_env_name = _os.environ.get("CONDA_DEFAULT_ENV")
    _BANNER = "=" * 72
    print()
    print(_BANNER)
    print("  PREFLIGHT: PyMC backend missing — PyMC steps will be SKIPPED")
    print(_BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    if _conda_env_name:
        print(f"  Conda env        : {_conda_env_name}")
    print("  Missing package  : 'pymc' (pymc_gp surrogate backend)")
    print()
    print("  HOW TO FIX — install a backend env and re-run this notebook on that kernel:")
    print("    conda env create -f environment.yml   # PyMC AND SBI; also covers T6")
    print("    conda activate py314_bayesmm")
    print("    python -m ipykernel install --user --name py314_bayesmm")
    print("    (PyMC alone: environment-pymc.yml -> py312_bayesmm_pymc)")
    print()
    print("  The rest of the notebook still runs; only the PyMC steps are skipped.")
    print(_BANNER)
    print()


### How the surrogate finds its training data (and why a surrogate tutorial starts by running a sweep)

The surrogate is never *handed* a dataset — it **looks one up, by path**:

- `tutorials/specs/model.toy.grid.json` -> `"storage": {"root": "tmp/tutorials/toy_store"}`
- `tutorials/specs/surrogate.toy.pymc_gp.json` -> `"dataset_ref": {"run_store_root": "tmp/tutorials/toy_store"}`

That matching string is the **entire join** between the sweep layer and the surrogate layer. There is no id, no registry entry, no pointer. Then `load_tabular_dataset` (`surrogates/dataset.py`):

1. reads **every** `sweep_rows.csv` under that root — a centralized store legitimately holds sweeps from several models;
2. skips any sweep whose header lacks the declared input columns (that is how a foreign model's sweep is ignored instead of crashing the fit);
3. keeps rows with `status == "success"`;
4. reduces each row's output array with `summary_config` — here `{"kind": "index", "index": 0}`, i.e. the surrogate learns `y[0] = a + b` **only**. The toy also emits `y[1] = a*b`, and that channel is thrown away. Step 6 puts it back.

Hence the next cell: without successful rows under that root there is nothing to fit.

In [ ]:
run_mm_cli('run', 'tutorials/specs/model.toy.grid.json')


## Step 2: Fit surrogate and list artifacts


`backend_config` in the spec is **MCMC over the surrogate's own four parameters**: NUTS draws 300 samples after 300 tuning steps in 1 chain, targeting an acceptance rate of 0.9. It fits `beta_a, beta_b, intercept, sigma` given the sweep rows.

It is *not* the sampling you will do in 7a-7c/T8 — that samples a *metamodel's* variables and is configured on the metamodel spec. Two layers, two samplers, two `draws` fields; keeping them apart is most of understanding this framework's structure.

`surrogate list` prints the **global** artifact registry — every fit from every spec you have ever run, so it grows across tutorials. `eval` does not use that list: it calls `find_latest_artifact_for_spec(spec.name)` and takes the newest artifact whose `spec_name` matches. Distinct spec `name`s never collide, which is what makes Step 6's second surrogate safe to fit alongside this one.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 2 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    run_mm_cli('surrogate', 'fit', 'tutorials/specs/surrogate.toy.pymc_gp.json')
    run_mm_cli('surrogate', 'list')


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 2 detail SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
    spec = None
else:
    import json

    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.storage import find_latest_artifact_for_spec
    from bayesian_metamodeling.surrogates import eval_surrogate
    from bayesian_metamodeling.surrogates.dataset import load_tabular_dataset

    def _abs(p):
        """Artifact paths are stored relative to the repo root."""
        p = Path(p)
        return p if p.is_absolute() else root / p

    SPEC_PATH = root / "tutorials/specs/surrogate.toy.pymc_gp.json"
    spec = SurrogateSpec.model_validate(json.loads(SPEC_PATH.read_text()))

    # Exactly what the fit consumed — the join described above, made visible.
    x_train, y_train, _ = load_tabular_dataset(spec)
    store_root = spec.dataset_ref["run_store_root"]
    sweep_csvs = sorted((root / store_root / "sweeps").rglob("sweep_rows.csv"))
    distinct = sorted({tuple(r) for r in x_train.tolist()})

    print(f"dataset_ref.run_store_root : {store_root}")
    print(f"sweep_rows.csv files read  : {len(sweep_csvs)}")
    print(f"training rows loaded       : x{x_train.shape}  y{y_train.shape}")
    print(f"distinct (a, b) points     : {len(distinct)} -> {distinct}")
    print(f"summary_config             : {spec.summary_config}  -> learning y[0] = a + b")
    for i in range(min(3, len(x_train))):
        print(f"  row {i}: a={x_train[i, 0]:.1f}  b={x_train[i, 1]:.1f}  ->  y={y_train[i, 0]:.1f}")

    if len(distinct) < x_train.shape[0]:
        print(
            f"\nNote: {x_train.shape[0]} rows but only {len(distinct)} distinct design points."
            "\nStep 1 appends a NEW sweep to the store every time you run it, and the loader reads"
            "\nall of them. Duplicated rows add no design information, but they do move sigma's"
            "\nposterior — one reason to read the widths this notebook prints rather than any"
            "\nnumber written in prose (here or anywhere else)."
        )

## Step 3: Evaluate on new inputs

`--n 200` sizes the *sample array* the command returns (look at `sample_shape` in the output). It does **not** affect the reported `mean` and `std`: `summary()` is computed in closed form from the posterior draws and ignores `n` entirely. Step 5 checks that claim by evaluating twice.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 3 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    run_mm_cli('surrogate', 'eval', 'tutorials/specs/surrogate.toy.pymc_gp.json', '--inputs', '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}', '--n', '200')


## Mini-lesson: prior, posterior, posterior predictive

Three Bayesian terms you will meet across this curriculum. Because this backend has only four parameters, you can name every one of them — which is a far better first Bayesian lesson than a hand-wave about function spaces.

- **Prior**: beliefs about the *parameters*, before data. Here, literally `beta_a, beta_b, intercept ~ Normal(0, 2)` and `sigma ~ HalfNormal(1)`. Note what the prior does *not* control: the shape is hard-wired to a plane by the choice of backend. The prior only says the plane's slopes are probably within a few units of zero.
- **Posterior**: beliefs about those same four numbers *after* conditioning on the sweep rows. Step 5 prints them, mean and sd.
- **Posterior predictive**: the distribution of the *output* `y` at a *new* input, integrating over the posterior — draw `(beta, intercept, sigma)` from the posterior, compute `mu = intercept + a*beta_a + b*beta_b`, add `Normal(0, sigma)`. This is what `eval_surrogate` summarizes.

And the width itself, straight out of `PymcPosteriorLinearModel.summary`:

```
std(x) = sqrt( Var_draws( mu(x) )  +  E_draws[ sigma^2 ] )
          \_______________/        \________________/
          parameter uncertainty      fitted residual scale
          depends on x               constant in x
```

Keep that decomposition in mind for the plot: the dots are posterior predictive means, the bars are those standard deviations, and only the first term under the square root knows where you queried.

**Common confusion: these errorbars are not confidence intervals.** They are posterior predictive standard deviations. Specifically they are **not**:

- a frequentist confidence interval ("95% of resampled estimates would fall in here") — wrong framework entirely; nothing here is resampled;
- the *simulator's* noise. The toy is deterministic, so the honest noise level is zero. `sigma` is the scale of everything the plane failed to explain — **misfit**, not measurement error. On a channel the model can represent exactly, `sigma` collapses toward zero (a `HalfNormal(1)` prior never lets it arrive); on a channel it cannot, `sigma` is large and that is the model telling you so. Step 6 is that case;
- a single number that tracks distance to the training data. Both terms exist, but `E[sigma^2]` does not depend on `a` or `b` at all, and `Var_draws(mu)` — which does — is often the smaller one. Which term dominates is an empirical question, and Step 5 answers it by printing both.

The practical consequence: **on this backend a narrow band is not evidence that you are near training data**, and a wide band is not evidence that you are far from it. A wide band means unexplained structure; a narrow one means the plane fit what it saw. Extrapolation is a separate question the width barely answers, which is exactly why Step 5 queries at `a = b = 20`.

## Step 4: Plot predictive mean and uncertainty (graphic)

Heads-up about the plot below: on this channel `y = a + b` is *precisely* the shape a plane can represent, so the fit is essentially exact, `sigma` collapses, and the errorbars come out microscopic — invisible on an axis spanning 0 to 4. The exact magnitude depends on your store, your seed and your PyMC version, so read it off the annotation and the printed list rather than remembering a number. Do not read "invisible bar" as "good surrogate": Step 6 runs the same backend on a channel where the bar is plainly visible *and* the mean is wrong.

### Predict before you plot

You just fitted a four-parameter plane to a linear function, and you are about to query four points it never saw, all *inside* the training box. Commit to answers first:

1. How close will the predicted means be to `a + b` — within 10%? 1%? machine precision?
2. Will the errorbars be visible on an axis that spans 0 to 4?
3. Will the bar at the query point closest to a training point be visibly narrower than the others?

Then read the plot and the printed numbers.

*(Expected: means correct to something like 1e-8 rather than 1e-2, because a linear model handed a linear function has nothing left to be wrong about; bars of order 1e-6 or smaller, invisible; and no — the four widths agree to about three digits, because they are dominated by one fitted `sigma` that does not depend on `a` or `b`. Step 5 takes the width apart and shows this.)*

**The point of the exercise:** a surrogate that is confidently wrong is worse than no surrogate. Here it is confidently *right* — but only because the truth happens to be exactly the shape the model can represent. Step 6 removes that coincidence and the same machinery produces a confident, wrong, and *visibly* uncertain answer.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 4 (plot) SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
    mean = None
    std = None
else:
    import csv

    import matplotlib.pyplot as plt

    # --- Query points (4 inputs the surrogate hasn't seen) ---
    inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)
    mean = np.asarray(result["summary"]["mean"], dtype=float)
    std = np.asarray(result["summary"]["std"], dtype=float)

    # --- Training overlay: read the newest sweep in the store the surrogate read from ---
    sweep_csv = max(
        (root / spec.dataset_ref["run_store_root"] / "sweeps").rglob("sweep_rows.csv"),
        key=lambda p: p.stat().st_mtime,
        default=None,
    )
    assert sweep_csv is not None, "no sweep_rows.csv under the store — Step 1 did not run"
    train_a, train_b, train_y = [], [], []
    with open(sweep_csv) as f:
        reader = csv.DictReader(f)
        # No fallback to the analytical formula: if the column is gone we want to hear
        # about it, not plot a synthetic curve that agrees with itself by construction.
        assert reader.fieldnames and "y__0" in reader.fieldnames, (
            f"sweep CSV {sweep_csv} has no y__0 column; store schema changed"
        )
        for row in reader:
            if row.get("status") != "success":
                continue
            train_a.append(float(row["a"]))
            train_b.append(float(row["b"]))
            # The toy outputs y = [a+b, a*b]; this surrogate trained on the first column.
            train_y.append(float(row["y__0"]))
    train_a = np.asarray(train_a)
    train_b = np.asarray(train_b)
    train_y = np.asarray(train_y)

    # Use `a + b` as x-axis (the truth is y = a + b, so predictions and training points
    # lie on one calibration curve).
    query_x = np.asarray(inputs["a"]) + np.asarray(inputs["b"])
    query_truth = query_x
    train_x = train_a + train_b

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    xx = np.linspace(0, 4, 50)
    ax.plot(xx, xx, "k:", alpha=0.5, label="analytical truth: y = a + b")
    ax.scatter(
        train_x, train_y, marker="x", s=70, c="tab:gray",
        alpha=0.7, label=f"training inputs (N={len(train_x)})", zorder=3,
    )
    ax.errorbar(
        query_x, mean, yerr=std, fmt="o", color="tab:blue",
        markersize=10, capsize=5, label="surrogate posterior predictive (mean ± std)", zorder=4,
    )
    ax.annotate(
        f"std ≈ {std[1]:.2e}\n(the whole band — sigma has collapsed)",
        xy=(query_x[1], mean[1]),
        xytext=(query_x[1] + 0.4, mean[1] - 0.6),
        fontsize=9, ha="left",
        arrowprops=dict(arrowstyle="->", color="tab:blue", alpha=0.6),
    )
    ax.set_title("pymc_gp surrogate vs analytical truth (with training overlay)")
    ax.set_xlabel("a + b  (query and training points share this axis)")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\nNumeric check (4 query points):")
    print(f"  predicted mean: {[float(f'{m:.6f}') for m in mean]}")
    print(f"  analytical y :  {[float(f'{t:.6f}') for t in query_truth]}")
    print(f"  std (each)   :  {[float(f'{s:.2e}') for s in std]}")
    print(f"  widest / narrowest std: {std.max() / std.min():.4f}x")

## Step 5: take the width apart, and leave the training box

Two claims from the prose above are now checkable, so check them rather than believing them.

1. **What the width is made of.** The cell reads the fitted artifact's own posterior draws, prints the four parameters, then rebuilds `std` from `Var_draws(mu)` and `E[sigma^2]` and asserts it matches what `eval_surrogate` reported. If the formula in the mini-lesson were wrong, this cell would fail.
2. **What happens outside the training box.** `a` and `b` were swept over `[0, 2]`. The probe queries `a = b = 0.5, 1, 5, 20` — the last is ten times the training range. Predict before you look: does the mean revert toward a prior mean, or continue the plane? Does the band open up?

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 5 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
    near_std = far_std = far_mean = None
else:
    # --- the posterior over the four parameters, read straight out of the artifact ---
    _aid, _apath = find_latest_artifact_for_spec(spec.name)
    _art = json.loads(_apath.read_text())
    _pl = json.loads(_abs(_art["backend_payload"]).read_text())
    W = np.asarray(_pl["posterior_weights"])[:, :, 0]   # (draws, n_inputs): the slopes
    C = np.asarray(_pl["posterior_bias"])[:, 0]         # (draws,): intercept
    S = np.asarray(_pl["posterior_sigma"])[:, 0]        # (draws,): sigma

    print(f"artifact {_aid}  ({_pl['model_type']}, {W.shape[0]} posterior draws)")
    print("posterior over the FOUR parameters this backend actually has:")
    for j, nm in enumerate(spec.inputs):
        print(f"  {'beta_' + nm:<10} mean={W[:, j].mean():+.6f}   sd={W[:, j].std():.2e}")
    print(f"  {'intercept':<10} mean={C.mean():+.6f}   sd={C.std():.2e}")
    print(f"  {'sigma':<10} mean={S.mean():.3e}   sd={S.std():.2e}")

    # --- inside the box, and 10x outside it ---
    probe = {"a": [0.5, 1.0, 5.0, 20.0], "b": [0.5, 1.0, 5.0, 20.0]}
    probe_res = eval_surrogate(spec=spec, inputs_payload=probe, n=200)
    p_mean = np.asarray(probe_res["summary"]["mean"], dtype=float)
    p_std = np.asarray(probe_res["summary"]["std"], dtype=float)

    Xq = np.column_stack([probe["a"], probe["b"]])
    mu_draws = Xq @ W.T + C[None, :]           # (N, draws): the linear mean per posterior draw
    var_param = mu_draws.var(axis=1)           # depends on the query point
    var_noise = float(np.mean(S**2))           # does not

    print(f"\n{'a=b':>5} {'truth a+b':>10} {'pred mean':>16} {'pred std':>11}"
          f" {'sqrt(Var mu)':>13} {'sqrt(E s^2)':>12}  (training box: a,b in [0,2])")
    for i, v in enumerate(probe["a"]):
        flag = "" if v <= 2.0 else "   <- outside"
        print(f"{v:5.1f} {2 * v:10.2f} {p_mean[i]:16.6f} {p_std[i]:11.3e}"
              f" {np.sqrt(var_param[i]):13.3e} {np.sqrt(var_noise):12.3e}{flag}")

    rebuilt = np.sqrt(var_param + var_noise)
    print(f"\nrebuilt std vs reported std, max relative gap: "
          f"{np.max(np.abs(rebuilt - p_std) / p_std):.1e}")
    assert np.allclose(rebuilt, p_std, rtol=1e-6), (
        "predictive std is NOT sqrt(Var_draws(mu) + E[sigma^2]) — the mini-lesson is wrong"
    )

    near_std, far_std, far_mean = float(p_std[1]), float(p_std[-1]), float(p_mean[-1])
    print(f"\nwidth at a=b=20 vs a=b=1      : {far_std / near_std:.2f}x  "
          f"(parameter uncertainty does grow — and is still ~{far_std:.0e})")
    print(f"error at a=b=20 (truth 40)    : {abs(far_mean - 40.0):.2e}  "
          f"— the plane keeps going, it does not revert to a prior mean")

    # --n sizes the returned sample array only; the summary is closed-form.
    small = eval_surrogate(spec=spec, inputs_payload=probe, n=20)
    print(f"\nsummary identical for n=20 and n=200: "
          f"{small['summary'] == probe_res['summary']}   "
          f"sample_shape {small['sample_shape']} vs {probe_res['sample_shape']}")

### Scientific checkpoint

- The mean at `a = b = 20` is 40 to several decimals. **This model does not revert to a prior mean off the data** — it extrapolates the plane, indefinitely, and would do so just as calmly if the truth curved away. That is the honest hazard of a linear surrogate, and the reason "the surrogate is uncertain out there" is not a claim you may make without checking.
- The width did grow going outside the box — `Var_draws(mu)` scales with the query — but from one microscopic number to another. The growth is real and useless: it would not survive being plotted.
- On this channel the correct prediction is right for a reason that will not generalize: the truth lies *inside* the model class. Nothing about the width told you that. The next step is what happens when it does not.

## Step 6: T1's actual question — the `product` channel

T1 asked: fitted on this 3x3 grid, would you trust a surrogate's `product = a*b` at `a = 0.5, b = 0.5`? So far T5 has only fitted the `sum` channel — the one case a plane cannot get wrong.

Change **one field** of the spec, `summary_config.index: 0 -> 1`, and the same backend, on the same nine runs, learns `y[1] = a*b` instead. Everything else — data, priors, sampler, query machinery — is held fixed, so whatever changes is caused by the target function alone. (A new `name` gives it its own artifact lineage; the spec is written under `tmp/`, which is gitignored.)

You can work out the answer before running it. Least squares fitting a plane `c + p*a + q*b` to `a*b` over the grid `{0,1,2}^2` gives exactly `a + b - 1`, and the residual it leaves behind is `(a-1)(b-1)`, whose RMS over the nine points is `2/3 = 0.667`. So predict: what should the fitted `sigma` come out as? What should the surrogate say at `a = b = 0.5`? What should it say at `a = b = 0`, where the truth is 0?

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 6 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
    prod_mean = prod_std = prod_truth = None
    prod_mae = None
else:
    # --- one field changed, written to a spec of its own ---
    prod_payload = json.loads(SPEC_PATH.read_text())
    prod_payload["name"] = "tutorial_toy_surrogate_pymc_product"
    prod_payload["summary_config"] = {"kind": "index", "index": 1}   # y[1] = a * b
    prod_spec_path = root / "tmp/tutorials/specs/surrogate.toy.pymc_gp.product.json"
    prod_spec_path.parent.mkdir(parents=True, exist_ok=True)
    prod_spec_path.write_text(json.dumps(prod_payload, indent=2, sort_keys=True))
    run_mm_cli("surrogate", "fit", str(prod_spec_path.relative_to(root)))

    prod_spec = SurrogateSpec.model_validate(prod_payload)

    # --- what plane did it settle on? ---
    _pid, _ppath = find_latest_artifact_for_spec(prod_spec.name)
    _ppl = json.loads(_abs(json.loads(_ppath.read_text())["backend_payload"]).read_text())
    Wp = np.asarray(_ppl["posterior_weights"])[:, :, 0]
    Cp = np.asarray(_ppl["posterior_bias"])[:, 0]
    Sp = np.asarray(_ppl["posterior_sigma"])[:, 0]
    print(f"\nfitted plane : y ~ {Cp.mean():+.3f} {Wp[:, 0].mean():+.3f}*a {Wp[:, 1].mean():+.3f}*b"
          f"   sigma = {Sp.mean():.3f}")
    print("predicted    : y ~ -1.000 +1.000*a +1.000*b   sigma = 0.667   (least squares, in closed form)")
    print("sigma is the RMS of the curvature (a-1)(b-1) the plane threw away: MISFIT, not simulator")
    print("noise — the toy is deterministic. With only 9 rows the prior pulls these in a little.")

    # Wide tolerances on purpose: with 9 rows the Normal(0,2) prior shrinks the estimates,
    # and how many duplicate sweeps sit in your store moves them too. A garbage fit still fails.
    assert 0.5 < Wp[:, 0].mean() < 1.5 and 0.5 < Wp[:, 1].mean() < 1.5, "slopes are not ~1"
    assert -1.6 < Cp.mean() < -0.4, "intercept is not ~ -1"

    # --- evaluate along the diagonal a = b, where the truth is a^2 ---
    grid = [round(float(v), 3) for v in np.linspace(0.0, 2.0, 9)]
    prod_inputs = {"a": grid, "b": grid}
    prod_res = eval_surrogate(spec=prod_spec, inputs_payload=prod_inputs, n=200)
    prod_mean = np.asarray(prod_res["summary"]["mean"], dtype=float)
    prod_std = np.asarray(prod_res["summary"]["std"], dtype=float)
    prod_truth = np.asarray(grid) * np.asarray(grid)
    prod_mae = float(np.mean(np.abs(prod_mean - prod_truth)))

    i_half = grid.index(0.5)
    print("\nT1's question, answered — product at a = b = 0.5:")
    print("  truth              : 0.2500")
    print(f"  surrogate          : {prod_mean[i_half]:+.4f} ± {prod_std[i_half]:.4f}")
    print(f"  MAE over this diagonal      : {prod_mae:.4f}  "
          f"(the exact plane would give mean((a-1)^2) = 0.4167)")
    print(f"  most negative predicted a*b : {prod_mean.min():+.4f}  (the truth is never negative)")
    print(f"  width, narrowest -> widest  : {prod_std.min():.3f} -> {prod_std.max():.3f}  "
          f"(vs ~{near_std:.0e} on the sum channel)")

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    aa = np.linspace(0, 2, 100)
    ax.plot(aa, aa * aa, "k:", label="analytical truth: y = a*b along a = b")
    ax.errorbar(
        grid, prod_mean, yerr=prod_std, fmt="o", color="tab:red", markersize=8, capsize=5,
        label="surrogate posterior predictive (mean ± std)", zorder=4,
    )
    ax.scatter(
        [0, 1, 2], [0, 1, 4], marker="x", s=70, c="tab:gray",
        label="training points on this diagonal (6 more lie off it)", zorder=3,
    )
    ax.axhline(0.0, color="0.6", lw=0.8)
    ax.set_title("Same backend, same 9 runs, product channel: a plane through a curve")
    ax.set_xlabel("a  (with b = a)")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

### Scientific checkpoint — and the answer to T1

- **No, you should not trust it.** At `a = b = 0.5` the truth is 0.25 and the surrogate reports roughly 0 — the exact plane gives `0.5 + 0.5 - 1 = 0` — with a width of order 1. The interval covers the truth, which is the model being *honest*, not *useful*: a band that wide over an output whose range is `[0, 4]` is a refusal to answer.
- **At `a = b = 0` it predicts a negative product.** No `a*b` is negative anywhere in the domain. A model class that cannot represent the sign structure of the function will violate it, and no amount of data fixes that — this is bias, not variance.
- **The width is the warning, and here it works** — but notice *why*. It is large because `sigma` absorbed the misfit, not because the query is far from training data. Same mechanism as the invisible bars in Step 4, opposite reading.
- **What this changes about Step 4.** The sum channel looked like a triumph. It was a tautology: a linear model fit to a linear function. Any claim you make about a surrogate's accuracy on a function inside its model class transfers to nothing.
- **When is this whole approach the wrong tool?** When the model class cannot contain the response surface and you have no diagnostic that says so. Here you did: `sigma` >> 0, plus a residual pattern with obvious structure. Fixes, in order of effort: engineer features (add an `a*b` column to the sweep and the plane becomes exact), use a genuinely nonlinear surrogate (`sbi_npe` in T6), or refuse to surrogate and pay for the simulator.

## Optional appendix: confidence check that the PyMC backend works

The cell below runs a single regression test from the package's own test suite. It's not part of the lesson — it's just a "is my PyMC install actually functional" smoke test you can run if anything earlier surprised you. Skip on first read.


In [ ]:
if not PYMC_AVAILABLE:
    print("Optional appendix SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'pymc_gp_backend_fit_sample_and_logprob')


## Recap: what T5 established

- A **surrogate** is a cheap probabilistic stand-in for an expensive model, trained from `sweep_rows.csv` — the same file T1 produced — and found by a **matching path string** (`storage.root` == `dataset_ref.run_store_root`), not by an id.
- **`pymc_gp` is Bayesian linear regression, not a Gaussian process.** Four parameters — `beta_a`, `beta_b`, `intercept`, `sigma` — whose posterior you printed and whose fitted plane you read off. Check what a backend id compiles to before reasoning from its name.
- The predictive width is `sqrt(Var_draws(mu) + E[sigma^2])`: parameter uncertainty (moves with the query) plus fitted residual scale (does not). You rebuilt it from the artifact and it matched.
- **`sigma` measures misfit, not simulator noise.** Microscopic on the sum channel, where the plane fits exactly (the exact value drifts with store size, seed and PyMC version — read the one your run printed); `2/3` on the product channel, which is exactly the RMS of the curvature `(a-1)(b-1)` a plane must discard.
- **This surrogate does not widen off the training box.** At `a = b = 20` — ten times the swept range — it reports 40 to several decimals, and the width, though it does grow, stays far too small to see. Confident extrapolation is a property of the model class you chose; a GP would do something else, and this is not one.
- Artifacts are **content-stamped and re-loadable**: each fit gets a fresh UUID directory recording `spec_digest`, `dataset_digest`, `seed` and dependency versions, and `find_latest_artifact_for_spec` resolves the newest by spec name. **T9** reuses this machinery on a sweep you design; module 7 and T8 deliberately use pre-built stub artifacts so the coupling lesson stays isolated from surrogate fitting.
- **Where this width actually gets used:** `bayesmm meta sample --method joint` evaluates a fitted surrogate's `log_prob`, so a narrow band pulls the coupled variable hard and a wide one lets it float. The default `--method propagate` does **not** evaluate surrogates at all — it draws from the priors and applies the coupling transforms. T7 draws that distinction; keep it in mind, because it decides whether today's lesson touches tomorrow's inference.

One sentence to carry forward: *a width tells you what the model failed to explain — read what it is made of before you let it stand in for the simulator.*

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: PyMC backend missing` | `pymc` absent from this kernel | `conda env create -f environment.yml`, then run this notebook on the `py314_bayesmm` kernel (it also carries SBI for T6). The main dev env ships without backends by design. |
| Fit takes minutes, or PyTensor compiles C code | PyTensor is building its C ops | Set `PYTENSOR_FLAGS=cxx=` for the C-less path. Slower per-sample, far faster to start, and deterministic across machines. |
| `No surrogate artifact produced` in the self-check | Step 2's fit never ran | Scroll up for the preflight banner. Skipped steps say so loudly. |
| `No surrogate artifact found for spec_name=tutorial_toy_surrogate_pymc_product` | Step 6's `fit` cell was skipped or failed | Re-run Step 6 top to bottom; `eval` only resolves artifacts whose `spec_name` matches. |
| Predictions form a plane no matter what the data does | The backend is linear in its inputs and cannot represent curvature | Expected, not a bug — that is Step 6. Add engineered columns to the sweep (an `a*b` input makes the plane exact), or switch to `sbi_npe` (T6). |
| The band barely changes across query points | `std = sqrt(Var_draws(mu) + E[sigma^2])`, and `E[sigma^2]` does not depend on the inputs at all | Expected. Query far outside the box (Step 5 uses `a = b = 20`) to make the parameter-uncertainty term visible; it is still tiny. |
| Your printed widths differ from a colleague's by 10x | Step 1 appends a sweep to the store on every run and the loader reads them all, so `N` differs; and on the sum channel `sigma` is chasing a degenerate zero | Expected, and harmless. Read the number your own notebook printed; never one from prose. |

**Next:** T6 fits a fundamentally different surrogate — SBI's neural posterior estimation — on this same data, so you can compare what a Bayesian linear model and a neural density estimator each believe about the same nine points. Step 6 above is the fair test to hold them to.

## Final check: T5 fitted a real surrogate, and the lesson is present

Resolves the artifact **for this notebook's spec** (`find_latest_artifact_for_spec`, not "newest file on disk" — you may have fitted others in this session), and asserts four things that a notebook which ran but taught nothing would fail:

1. the artifact is a `pymc_gp` fit with a payload on disk and the configured number of posterior draws;
2. **sum channel** — mean matches `a + b` and the width has collapsed;
3. **extrapolation** — at `a = b = 20` the mean is still 40 and the width still tiny, i.e. the model does *not* revert to a prior mean, and the parameter-uncertainty term did grow;
4. **product channel (Step 6)** — the same backend is badly wrong (`MAE` large), predicts a negative product, and says so with a width orders of magnitude larger than in (2).

(2) alone is a tautology — a linear model fitting a linear function — which is why (3) and (4) are here.

In [ ]:
if not PYMC_AVAILABLE:
    print(f"\n[T5 self-check OK] PyMC steps skipped per preflight (PYMC_AVAILABLE={PYMC_AVAILABLE}).")
else:
    # 1. the artifact THIS spec resolves to
    _aid, _apath = find_latest_artifact_for_spec(spec.name)
    _payload = json.loads(_apath.read_text())
    assert _payload["backend"] == "pymc_gp", f"artifact backend={_payload['backend']}"
    assert _payload["spec_name"] == spec.name, "resolved an artifact from another spec"
    assert _abs(_payload["backend_payload"]).exists(), "artifact has no fitted backend payload"

    _inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    _res = eval_surrogate(spec=spec, inputs_payload=_inputs, n=200)
    _mean = np.asarray(_res["summary"]["mean"], dtype=float)
    _std = np.asarray(_res["summary"]["std"], dtype=float)
    _truth = np.asarray(_inputs["a"]) + np.asarray(_inputs["b"])
    _expected_draws = spec.backend_config["draws"] * spec.backend_config["chains"]
    assert _res["summary"]["posterior_draws"] == _expected_draws, (
        f"{_res['summary']['posterior_draws']} posterior draws, spec asks for {_expected_draws}"
    )

    # 2. sum channel: the truth is inside the model class, so it is nailed and sigma collapses
    _mae = float(np.mean(np.abs(_mean - _truth)))
    assert _mae < 1e-3, f"sum-channel MAE={_mae:.3e} — the linear fit is broken"
    assert _std.max() < 1e-2, f"sum-channel width {_std.max():.3e} — sigma did not collapse"

    # 3. off the training box it extrapolates the plane instead of reverting to a prior mean
    assert far_mean is not None, "Step 5 did not run"
    assert abs(far_mean - 40.0) < 0.05, f"mean at a=b=20 is {far_mean:.4f}, not ~40"
    assert far_std < 1e-2, f"width at a=b=20 is {far_std:.3e} — unexpectedly large"
    assert far_std > near_std, "parameter uncertainty did not grow away from the data"

    # 4. the contrast that makes the width worth reading (Step 6)
    assert prod_mean is not None, "Step 6 did not run — Steps 1-5 alone do not cover T5's lesson"
    assert prod_mae > 0.2, f"product-channel MAE={prod_mae:.3f} — a plane should NOT fit a*b"
    assert prod_std.min() > 0.2, f"product-channel width {prod_std.min():.3f} — sigma should be ~2/3"
    assert prod_mean.min() < 0.0, "the plane should predict a negative product near the origin"
    assert prod_std.min() > 100 * _std.max(), "the two channels' widths should differ by orders"

    print(
        f"\n[T5 self-check OK] sum channel: MAE={_mae:.2e}, width<={_std.max():.1e}; "
        f"a=b=20 -> {far_mean:.4f} (width {far_std:.1e}); "
        f"product channel: MAE={prod_mae:.3f}, width>={prod_std.min():.3f}, "
        f"min mean {prod_mean.min():+.3f}; artifact {_apath}"
    )